# Retrieval method experiments

Compare retrieval methods on the same grounded cases and production text index.

Methods: dense, dense Qdrant, TF-IDF, hybrid RRF, dense+rerank, hybrid+rerank, and multi-query (Gemini variants + dense RRF).

Uses production defaults from doc_rag.5_embeddings and doc_rag.7_retrieval.


## 1. Setup

In [1]:
import gc
import importlib
import json
import re
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from ftfy import fix_text
from IPython.display import display
from qdrant_client import QdrantClient
from sentence_transformers import CrossEncoder

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "doc_notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

retrieval = importlib.reload(importlib.import_module("doc_rag.7_retrieval"))
embeddings_module = importlib.reload(importlib.import_module("doc_rag.5_embeddings"))

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this experiment.")

DEVICE = "cuda"
MODEL_NAME = embeddings_module.DEFAULT_MODEL_NAME
RERANKER_NAME = retrieval.DEFAULT_RERANKER_MODEL
COLLECTION_NAME = retrieval.DEFAULT_COLLECTION_NAME
TOP_K = retrieval.DEFAULT_TOP_K
CANDIDATE_K = retrieval.DEFAULT_CANDIDATE_K
GEMINI_MODEL = "gemini-3.6-flash"
GEMINI_VARIANT_COUNT = 3

EMBEDDING_DIR = (
    embeddings_module.DEFAULT_EMBEDDING_ROOT
    / embeddings_module._safe_model_name(MODEL_NAME)
)
PROCESSED_DIR = PROJECT_ROOT / "data" / "rag" / "processed_documents"


OUTPUT_DIR = PROJECT_ROOT / "data" / "rag" / "experiments" / "retrieval_evaluation"
CASES_FILE = OUTPUT_DIR / "text_method_selection_queries.json"
VARIANTS_FILE = OUTPUT_DIR / "gemini_query_variants.json"
RESULTS_FILE = OUTPUT_DIR / "retrieval_method_results.json"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("GPU:", torch.cuda.get_device_name(0))
print("Embedding model:", MODEL_NAME)
print("Reranker:", RERANKER_NAME)
print("Collection:", COLLECTION_NAME)
print("Embedding dir:", EMBEDDING_DIR)
print("Gemini variants cache:", VARIANTS_FILE)


GPU: NVIDIA GeForce MX570 A
Embedding model: BAAI/bge-m3
Reranker: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Collection: gatherly_document_text_bge_m3_v1
Embedding dir: C:\Users\User\Desktop\inmind\gatherly_rag\data\rag\embeddings\BAAI__bge-m3
Gemini variants cache: C:\Users\User\Desktop\inmind\gatherly_rag\data\rag\experiments\retrieval_evaluation\gemini_query_variants.json


## 2. Load production chunks, embeddings, and cases

In [2]:
if not (EMBEDDING_DIR / "embeddings.npy").is_file():
    raise FileNotFoundError(
        f"Missing embeddings at {EMBEDDING_DIR}. "
        "Run the text pipeline for the production model first."
    )

embedding_matrix = np.load(EMBEDDING_DIR / "embeddings.npy", allow_pickle=False)
saved_ids = json.loads(
    (EMBEDDING_DIR / "chunks.json").read_text(encoding="utf-8")
)["chunk_ids"]

chunk_rows = []
for path in PROCESSED_DIR.glob("*/chunks.json"):
    payload = json.loads(path.read_text(encoding="utf-8"))
    chunk_rows.extend(payload.get("chunks", []))
all_chunks = pd.DataFrame(chunk_rows)
if all_chunks.empty or all_chunks["chunk_id"].duplicated().any():
    raise ValueError("Processed chunk artifacts are empty or contain duplicate IDs.")

chunks_by_id = all_chunks.set_index("chunk_id", drop=False)
missing_ids = [chunk_id for chunk_id in saved_ids if chunk_id not in chunks_by_id.index]
if missing_ids:
    raise ValueError(f"Missing {len(missing_ids)} embedding-aligned chunks.")

chunks_df = chunks_by_id.loc[saved_ids].reset_index(drop=True)
if len(chunks_df) != embedding_matrix.shape[0]:
    raise ValueError("Chunk and embedding counts do not match.")

raw_cases = json.loads(CASES_FILE.read_text(encoding="utf-8"))
if isinstance(raw_cases, dict) and "cases" in raw_cases:
    raw_cases = raw_cases["cases"]
if not isinstance(raw_cases, list) or not raw_cases:
    raise ValueError(f"No cases in {CASES_FILE}")

LANG_MAP = {"en": "English", "fr": "French", "ar": "Arabic",
            "english": "English", "french": "French", "arabic": "Arabic"}

ALLOWED_LANGUAGES = {"Arabic", "English", "French"}

cases = []
for case in raw_cases:
    if case.get("answerable") is False:
        continue
    query_type = str(case.get("query_type", "text")).casefold()
    if query_type not in {"text", "text_image", ""}:
        continue

    expected_file = case.get("expected_file") or case.get("expected_document")
    if not expected_file:
        raise ValueError(f"Case {case.get('id')} missing expected document")

    normalized = dict(case)
    normalized["query"] = fix_text(str(case["query"]))
    normalized["expected_file"] = fix_text(str(expected_file))
    normalized["expected_pages"] = list(case.get("expected_pages") or [])

    lang = str(case.get("language", "")).strip()
    normalized["language"] = LANG_MAP.get(lang.casefold(), lang if lang in ALLOWED_LANGUAGES else "")
    cases.append(normalized)




FRENCH_QUERY_RE = re.compile(
    r"(?i)\b("
    r"quelles?|qui|quel|qu'est-ce|dans quels|s'occupe|"
    r"comprennent|prévoir|événementielle|organisateurs?"
    r")\b"
)


def case_language(case: dict) -> str:
    """Prefer explicit case label; otherwise infer from id/script/French cues."""
    labeled = str(case.get("language", "")).strip()
    if labeled in ALLOWED_LANGUAGES:
        return labeled

    case_id = str(case.get("id", ""))
    query = str(case.get("query", ""))

    if case_id.startswith("arabic_") or any(
        "\u0600" <= ch <= "\u06FF" for ch in query
    ):
        return "Arabic"
    if FRENCH_QUERY_RE.search(query) or any(
        ch in query for ch in "àâäéèêëïîôùûüçÀÂÄÉÈÊËÎÏÔÙÛÜÇ"
    ):
        return "French"
    return "English"


for case in cases:
    case["language"] = case_language(case)

print("Chunks:", len(chunks_df))
print("Embedding shape:", embedding_matrix.shape)
print("Cases:", len(cases))
display(pd.Series([c["language"] for c in cases]).value_counts().rename("cases"))
display(chunks_df.groupby("file_name").size().rename("chunks").head(10))

Chunks: 775
Embedding shape: (775, 1024)
Cases: 38


English    33
French      4
Arabic      1
Name: cases, dtype: int64

file_name
Boho_Wedding.pdf                    61
Celestial_Wedding.pdf               14
Checklists.pdf                      17
Classic_Wedding.pdf                 20
Enchanted_Forest_Wedding.pdf        44
Garden_Wedding.pdf                  30
Guideline_Sustainable_Event.pdf     70
Modern_Wedding.pdf                  61
Moody_Wedding.pdf                   20
Organiser_un_evenement_deAaZ.pdf    82
Name: chunks, dtype: int64

## 3. Load embedding model, TF-IDF, Qdrant, reranker

In [3]:
model_kwargs = {"torch_dtype": torch.float16}
embedding_model = embeddings_module.load_embedding_model(
    MODEL_NAME,
    device=DEVICE,
    local_files_only=True,
    model_kwargs=model_kwargs,
)
tfidf_index = retrieval.build_tfidf_index(chunks_df)

qdrant_client = QdrantClient(url="http://localhost:6333", timeout=60)
qdrant_info = qdrant_client.get_collection(COLLECTION_NAME)
if qdrant_info.points_count != len(chunks_df):
    raise ValueError(
        f"Qdrant points ({qdrant_info.points_count}) != chunks ({len(chunks_df)})."
    )

reranker = CrossEncoder(
    RERANKER_NAME,
    device=DEVICE,
    local_files_only=True,
    model_kwargs=model_kwargs,
)

print("Device:", DEVICE)
print("TF-IDF shape:", tfidf_index["matrix"].shape)
print("Qdrant points:", qdrant_info.points_count)
print("Reranker ready:", RERANKER_NAME)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Device: cuda
TF-IDF shape: (775, 52320)
Qdrant points: 775
Reranker ready: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1


## 4. Generate or reuse Gemini query variants


In [4]:
load_dotenv(PROJECT_ROOT / ".env")
api_key = os.getenv("GEMINI_API_KEY")
if not api_key and not VARIANTS_FILE.is_file():
    raise ValueError("Set GEMINI_API_KEY in .env")

gemini_client = None
if api_key:
    from google import genai
    gemini_client = genai.Client(api_key=api_key)

query_variants = retrieval.get_or_create_gemini_query_variants(
    [case["query"] for case in cases],
    gemini_client,
    cache_file=VARIANTS_FILE,
    gemini_model=GEMINI_MODEL,
    variant_count=GEMINI_VARIANT_COUNT,
)



In [5]:
sample_items = list(query_variants.items())[:2]

for original, variants in sample_items:
    paraphrases = variants[1:]  # skip original at index 0
    
    print("ORIGINAL:", original)
    print("PARAPHRASES:")
    for i, text in enumerate(paraphrases, start=1):
        print(f"  {i}. {text}")
    print("\n\n\n")

ORIGINAL: Pour un événement corporate, qui définit l'identité visuelle et coordonne photographes, graphistes et réalisateurs une fois le concept validé ?
PARAPHRASES:
  1. Une fois le concept validé pour un événement d'entreprise, quelle personne établit l'identité visuelle et supervise graphistes, photographes et réalisateurs ?
  2. Qui prend en charge l'identité visuelle et la coordination des réalisateurs, graphistes et photographes après la validation du concept d'un événement corporate ?
  3. Après validation du concept d'un événement d'entreprise, quel rôle définit l'identité visuelle et coordonne les graphistes, photographes et réalisateurs ?




ORIGINAL: Des participants malentendants doivent suivre les débats en direct. Quel procédé technique le document décrit-il, et en quoi diffère-t-il de la reconnaissance vocale ?
PARAPHRASES:
  1. Quel procédé technique est mentionné dans le document pour permettre aux participants malentendants de suivre les débats en direct, et quelle 

## 5. Run methods on the same cases


In [6]:
methods = {
    "dense": lambda q: retrieval.retrieve_dense(
        q, embedding_model, embedding_matrix, chunks_df,
        top_k=TOP_K, model_name=MODEL_NAME,
    ),
    "dense_qdrant": lambda q: retrieval.retrieve_dense_qdrant(
        q, qdrant_client, embedding_model,
        collection_name=COLLECTION_NAME, top_k=TOP_K, model_name=MODEL_NAME,
    ),
    "tfidf": lambda q: retrieval.retrieve_tfidf(
        q, tfidf_index, chunks_df, top_k=TOP_K,
    ),
    "hybrid_rrf": lambda q: retrieval.retrieve_hybrid(
        q, embedding_model, embedding_matrix, chunks_df, tfidf_index,
        top_k=TOP_K, candidate_k=CANDIDATE_K, model_name=MODEL_NAME,
    ),
    "dense_reranked": lambda q: retrieval.retrieve_dense_reranked(
        q, embedding_model, embedding_matrix, chunks_df, reranker,
        top_k=TOP_K, candidate_k=CANDIDATE_K, model_name=MODEL_NAME,
    ),
    "hybrid_reranked": lambda q: retrieval.retrieve_hybrid_reranked(
        q, embedding_model, embedding_matrix, chunks_df, tfidf_index, reranker,
        top_k=TOP_K, candidate_k=CANDIDATE_K, rerank_k=CANDIDATE_K,
        model_name=MODEL_NAME,
    ),
    "multi_query": lambda q: retrieval.retrieve_multi_query(
        query_variants[q], embedding_model, embedding_matrix, chunks_df,
        top_k=TOP_K, candidate_k=CANDIDATE_K, model_name=MODEL_NAME,
    ),
}

detail_rows = []
case_rows = []

for method_name, retrieve_fn in methods.items():
    print("Running:", method_name)
    for case in cases:
        started = time.perf_counter()
        results = retrieve_fn(case["query"])
        elapsed_ms = (time.perf_counter() - started) * 1000

        relevant_rank = None
        for _, row in results.iterrows():
            is_relevant = (
                fix_text(str(row["file_name"])) == case["expected_file"]
                and int(row["page_number"]) in case["expected_pages"]
            )
            if is_relevant and relevant_rank is None:
                relevant_rank = int(row["rank"])
            detail_rows.append({
                "method": method_name,
                "case_id": case["id"],
                "language": case["language"],
                "query": case["query"],
                "expected_file": case["expected_file"],
                "expected_pages": case["expected_pages"],
                "rank": int(row["rank"]),
                "score": float(row["score"]),
                "chunk_id": str(row["chunk_id"]),
                "file_name": fix_text(str(row["file_name"])),
                "page_number": int(row["page_number"]),
                "text": str(row["text"]),
                "is_relevant": bool(is_relevant),
            })

        case_rows.append({
            "method": method_name,
            "case_id": case["id"],
            "language": case["language"],
            "relevant_rank": relevant_rank,
            "latency_ms": elapsed_ms,
        })

case_results = pd.DataFrame(case_rows)
detailed_results = pd.DataFrame(detail_rows)
print("Completed evaluations:", len(case_results))


Running: dense
Running: dense_qdrant
Running: tfidf
Running: hybrid_rrf
Running: dense_reranked
Running: hybrid_reranked
Running: multi_query
Completed evaluations: 266


## 6. Metrics and recommendation


In [7]:
def summarize(group: pd.DataFrame) -> pd.Series:
    ranks = group["relevant_rank"]
    return pd.Series({
        "cases": len(group),
        "Recall@1": ranks.le(1).fillna(False).mean(),
        "Recall@3": ranks.le(3).fillna(False).mean(),
        "Recall@5": ranks.le(5).fillna(False).mean(),
        "MRR@5": ranks.apply(
            lambda rank: 0.0 if pd.isna(rank) or rank > 5 else 1.0 / rank
        ).mean(),
        "failures": ranks.isna().sum(),
        "mean_latency_ms": group["latency_ms"].mean(),
    })


overall = (
    case_results.groupby("method", sort=False)
    .apply(summarize, include_groups=False)
    .reset_index()
)
by_language = (
    case_results.groupby(["method", "language"], sort=False)
    .apply(summarize, include_groups=False)
    .reset_index()
)

language_floor = (
    by_language.groupby("method")["Recall@5"].min().rename("min_language_Recall@5")
)
ranking = overall.merge(language_floor, on="method")
ranking = ranking.sort_values(
    ["min_language_Recall@5", "Recall@5", "MRR@5", "Recall@1", "mean_latency_ms"],
    ascending=[False, False, False, False, True],
).reset_index(drop=True)
ranking.insert(0, "selection_rank", np.arange(1, len(ranking) + 1))
recommended_method = ranking.iloc[0]["method"]

display(overall.sort_values(["Recall@5", "MRR@5", "Recall@1"], ascending=False))
display(by_language.sort_values(["language", "Recall@5", "MRR@5"], ascending=[True, False, False]))
display(ranking)
print("Recommended retrieval method:", recommended_method)

,method,cases,Recall@1,Recall@3,Recall@5,MRR@5,failures,mean_latency_ms
5,hybrid_reranked,38.0,0.552632,0.815789,0.947368,0.705702,2.0,1628.272063
0,dense,38.0,0.578947,0.789474,0.921053,0.710088,3.0,60.004261
1,dense_qdrant,38.0,0.578947,0.789474,0.921053,0.710088,3.0,54.790032
4,dense_reranked,38.0,0.526316,0.815789,0.894737,0.677632,4.0,1701.984371
3,hybrid_rrf,38.0,0.526316,0.842105,0.868421,0.673246,5.0,39.228311
6,multi_query,38.0,0.605263,0.789474,0.842105,0.700439,6.0,115.777634
2,tfidf,38.0,0.342105,0.631579,0.631579,0.464912,14.0,3.522066


,method,language,cases,Recall@1,Recall@3,Recall@5,MRR@5,failures,mean_latency_ms
2,dense,Arabic,1.0,1.000000,1.000000,1.000000,1.000000,0.0,30.472900
5,dense_qdrant,Arabic,1.0,1.000000,1.000000,1.000000,1.000000,0.0,40.984900
11,hybrid_rrf,Arabic,1.0,1.000000,1.000000,1.000000,1.000000,0.0,38.746900
14,dense_reranked,Arabic,1.0,1.000000,1.000000,1.000000,1.000000,0.0,2454.627600
17,hybrid_reranked,Arabic,1.0,1.000000,1.000000,1.000000,1.000000,0.0,2067.777000
20,multi_query,Arabic,1.0,1.000000,1.000000,1.000000,1.000000,0.0,110.189300
8,tfidf,Arabic,1.0,0.000000,0.000000,0.000000,0.000000,1.0,3.402900
16,hybrid_reranked,English,33.0,0.515152,0.787879,0.939394,0.676263,2.0,1637.162036
1,dense,English,33.0,0.545455,0.757576,0.909091,0.681313,3.0,36.556470
4,dense_qdrant,English,33.0,0.545455,0.757576,0.909091,0.681313,3.0,49.494658


,selection_rank,method,cases,Recall@1,Recall@3,Recall@5,MRR@5,failures,mean_latency_ms,min_language_Recall@5
0,1,hybrid_reranked,38.0,0.552632,0.815789,0.947368,0.705702,2.0,1628.272063,0.939394
1,2,dense_qdrant,38.0,0.578947,0.789474,0.921053,0.710088,3.0,54.790032,0.909091
2,3,dense,38.0,0.578947,0.789474,0.921053,0.710088,3.0,60.004261,0.909091
3,4,dense_reranked,38.0,0.526316,0.815789,0.894737,0.677632,4.0,1701.984371,0.878788
4,5,hybrid_rrf,38.0,0.526316,0.842105,0.868421,0.673246,5.0,39.228311,0.848485
5,6,multi_query,38.0,0.605263,0.789474,0.842105,0.700439,6.0,115.777634,0.818182
6,7,tfidf,38.0,0.342105,0.631579,0.631579,0.464912,14.0,3.522066,0.000000


Recommended retrieval method: hybrid_reranked


## 6b. After winner lock: theme reorder on the winning method

Compare the selected method (`recommended_method`, expected: `hybrid_reranked`) with the same retrieve + optional theme-document reorder (same top-k chunks; order only changes when the query names a theme PDF).

In [8]:
from IPython.display import Markdown

WINNER = str(recommended_method)
if WINNER != "hybrid_reranked":
    print(
        f"Warning: recommended_method={WINNER!r}. "
        "Theme compare still runs on hybrid_reranked (production text lock)."
    )
    WINNER = "hybrid_reranked"

THEME_METHOD = f"{WINNER}_theme_reorder"
theme_registry = retrieval.build_theme_document_registry(chunks_df)
print(f"Winner base method: {WINNER}")
print(f"Theme registry size: {len(theme_registry)}")

def first_relevant_rank(case, frame) -> int | None:
    relevant_rank = None
    for _, row in frame.iterrows():
        is_relevant = (
            fix_text(str(row["file_name"])) == case["expected_file"]
            and int(row["page_number"]) in case["expected_pages"]
        )
        if is_relevant and relevant_rank is None:
            relevant_rank = int(row["rank"])
    return relevant_rank


theme_case_rows = []
theme_detail_rows = []
theme_triggered = 0
same_chunk_set = 0
order_changed = 0

for case in cases:
    started = time.perf_counter()
    base = retrieval.retrieve_hybrid_reranked(
        case["query"],
        embedding_model,
        embedding_matrix,
        chunks_df,
        tfidf_index,
        reranker,
        top_k=TOP_K,
        candidate_k=CANDIDATE_K,
        rerank_k=CANDIDATE_K,
        model_name=MODEL_NAME,
    )
    base_ms = (time.perf_counter() - started) * 1000

    themed = retrieval.apply_optional_theme_rerank(
        case["query"],
        base.copy(),
        theme_registry,
        top_k=TOP_K,
        boost=retrieval.DEFAULT_THEME_RERANK_LAMBDA,
    )
    total_ms = (time.perf_counter() - started) * 1000

    applied = bool(len(themed) and bool(themed["theme_rerank_applied"].iloc[0]))
    if applied:
        theme_triggered += 1

    base_ids = [str(x) for x in base["chunk_id"].tolist()]
    theme_ids = [str(x) for x in themed["chunk_id"].tolist()]
    if set(base_ids) == set(theme_ids):
        same_chunk_set += 1
    if base_ids != theme_ids:
        order_changed += 1

    base_rank = first_relevant_rank(case, base)
    theme_rank = first_relevant_rank(case, themed)

    for method_name, rel_rank, latency, frame in [
        (WINNER, base_rank, base_ms, base),
        (THEME_METHOD, theme_rank, total_ms, themed),
    ]:
        theme_case_rows.append({
            "method": method_name,
            "case_id": case["id"],
            "language": case["language"],
            "query": case["query"],
            "expected_file": case["expected_file"],
            "theme_rerank_applied": applied if method_name == THEME_METHOD else False,
            "matched_theme_files": (
                list(themed["theme_matched_files"].iloc[0])
                if method_name == THEME_METHOD and applied
                else []
            ),
            "relevant_rank": rel_rank,
            "latency_ms": latency,
        })
        for _, row in frame.iterrows():
            is_relevant = (
                fix_text(str(row["file_name"])) == case["expected_file"]
                and int(row["page_number"]) in case["expected_pages"]
            )
            theme_detail_rows.append({
                "method": method_name,
                "case_id": case["id"],
                "language": case["language"],
                "query": case["query"],
                "expected_file": case["expected_file"],
                "expected_pages": case["expected_pages"],
                "rank": int(row["rank"]),
                "score": float(row["score"]),
                "chunk_id": str(row["chunk_id"]),
                "file_name": fix_text(str(row["file_name"])),
                "page_number": int(row["page_number"]),
                "text": str(row["text"]),
                "is_relevant": bool(is_relevant),
                "theme_boosted": (
                    bool(row["theme_boosted"]) if "theme_boosted" in frame.columns else False
                ),
            })

theme_case_results = pd.DataFrame(theme_case_rows)
theme_detailed_results = pd.DataFrame(theme_detail_rows)

print(f"Cases: {len(cases)}")
print(f"Theme stage triggered: {theme_triggered}/{len(cases)}")
print(f"Same chunk set: {same_chunk_set}/{len(cases)}")
print(f"Order changed: {order_changed}/{len(cases)}")

theme_overall = (
    theme_case_results.groupby("method", sort=False)
    .apply(summarize, include_groups=False)
    .reset_index()
)
display(Markdown("### Winner vs winner + theme reorder"))
display(theme_overall)

triggered_ids = set(
    theme_case_results.loc[
        (theme_case_results["method"] == THEME_METHOD)
        & (theme_case_results["theme_rerank_applied"]),
        "case_id",
    ].tolist()
)
triggered_compare = theme_case_results[
    theme_case_results["case_id"].isin(triggered_ids)
]
if len(triggered_compare):
    theme_triggered_overall = (
        triggered_compare.groupby("method", sort=False)
        .apply(summarize, include_groups=False)
        .reset_index()
    )
    display(Markdown("### Theme-triggered subset only"))
    display(theme_triggered_overall)
else:
    print("No theme-triggered queries in this case set.")

flips = []
for case_id in sorted(triggered_ids):
    sub = theme_case_results[theme_case_results["case_id"] == case_id]
    b = sub[sub["method"] == WINNER].iloc[0]
    t = sub[sub["method"] == THEME_METHOD].iloc[0]
    if b["relevant_rank"] != t["relevant_rank"]:
        flips.append({
            "case_id": case_id,
            "query": b["query"],
            "expected_file": b["expected_file"],
            "winner_rank": b["relevant_rank"],
            "theme_rank": t["relevant_rank"],
            "matched_theme_files": t["matched_theme_files"],
        })
display(Markdown("### relevant_rank changes (triggered only)"))
display(pd.DataFrame(flips) if flips else pd.DataFrame({"note": ["no rank changes"]}))

Winner base method: hybrid_reranked
Theme registry size: 12
Cases: 38
Theme stage triggered: 12/38
Same chunk set: 38/38
Order changed: 5/38


### Winner vs winner + theme reorder

,method,cases,Recall@1,Recall@3,Recall@5,MRR@5,failures,mean_latency_ms
0,hybrid_reranked,38.0,0.552632,0.815789,0.947368,0.705702,2.0,1634.363724
1,hybrid_reranked_theme_reorder,38.0,0.552632,0.842105,0.947368,0.712281,2.0,1637.664703


### Theme-triggered subset only

,method,cases,Recall@1,Recall@3,Recall@5,MRR@5,failures,mean_latency_ms
0,hybrid_reranked,12.0,0.583333,0.833333,0.916667,0.729167,1.0,1452.032450
1,hybrid_reranked_theme_reorder,12.0,0.583333,0.916667,0.916667,0.750000,1.0,1458.650783


### relevant_rank changes (triggered only)

,case_id,query,expected_file,winner_rank,theme_rank,matched_theme_files
0,SEL_25,"For a classic wedding color scheme, which two ...",Classic_Wedding.pdf,4.0,2.0,[Classic_Wedding.pdf]
1,SEL_33,Which tropical blooms are recommended for bouq...,Tropical_Wedding.pdf,NaN,NaN,[Tropical_Wedding.pdf]


In [9]:
THEME_COMPARE_FILE = OUTPUT_DIR / "hybrid_theme_reorder_compare.json"
theme_payload = {
    "winner_method": WINNER,
    "theme_method": THEME_METHOD,
    "theme_registry_size": len(theme_registry),
    "n_cases": len(cases),
    "theme_triggered": theme_triggered,
    "same_chunk_set": same_chunk_set,
    "order_changed": order_changed,
    "overall": theme_overall.to_dict(orient="records"),
    "case_results": theme_case_results.where(pd.notna(theme_case_results), None).to_dict(
        orient="records"
    ),
}
THEME_COMPARE_FILE.write_text(
    json.dumps(theme_payload, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("Saved:", THEME_COMPARE_FILE)

Saved: C:\Users\User\Desktop\inmind\gatherly_rag\data\rag\experiments\retrieval_evaluation\hybrid_theme_reorder_compare.json


## 7. Failures and save


In [10]:
failures = case_results[case_results["relevant_rank"].isna()].sort_values(
    ["method", "language", "case_id"]
)
display(failures)

payload = {
    "schema_version": 1,
    "configuration": {
        "embedding_model": MODEL_NAME,
        "reranker_model": RERANKER_NAME,
        "collection": COLLECTION_NAME,
        "gemini_model": GEMINI_MODEL,
        "gemini_variant_count": GEMINI_VARIANT_COUNT,
        "top_k": TOP_K,
        "candidate_k": CANDIDATE_K,
        "case_count": len(cases),
    },
    "recommended_method": recommended_method,
    "ranking": ranking.to_dict(orient="records"),
    "overall_metrics": overall.to_dict(orient="records"),
    "language_metrics": by_language.to_dict(orient="records"),
    "case_results": case_results.where(pd.notna(case_results), None).to_dict(
        orient="records"
    ),
    "retrieved_results": detailed_results.to_dict(orient="records"),
}
RESULTS_FILE.write_text(
    json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Saved:", RESULTS_FILE)


,method,case_id,language,relevant_rank,latency_ms
21,dense,SEL_25,English,NaN,32.5487
29,dense,SEL_33,English,NaN,30.7165
35,dense,SEL_39,English,NaN,32.5729
59,dense_qdrant,SEL_25,English,NaN,45.9764
67,dense_qdrant,SEL_33,English,NaN,48.4754
73,dense_qdrant,SEL_39,English,NaN,70.2261
158,dense_reranked,SEL_07,English,NaN,2429.4014
181,dense_reranked,SEL_33,English,NaN,1431.0672
184,dense_reranked,SEL_36,English,NaN,1446.1937
187,dense_reranked,SEL_39,English,NaN,1619.5697


Saved: C:\Users\User\Desktop\inmind\gatherly_rag\data\rag\experiments\retrieval_evaluation\retrieval_method_results.json


## 8. Cleanup


In [11]:
del embedding_model, reranker
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
print("CUDA allocated MB:", round(torch.cuda.memory_allocated() / 1024**2, 2))
print("Cleanup complete.")

CUDA allocated MB: 9.12
Cleanup complete.
